# NEXTBUY: Business Insights

## Part 3: Business Insights Analysis

This notebook answers 8 business questions with visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    import seaborn as sns
except ImportError:
    sns = None

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Load Data

In [ ]:
PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data.parquet"
fallback_file = Path("full_data.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError("Processed data not found. Run preprocessing first.")

print(f"Loaded: {full_data.shape}")

---

## Question 1: Which products should be emphasized on Saturdays?

In [ ]:
# Saturday = 6 in our data
saturday_orders = full_data[full_data['order_dow'] == 6]
other_orders = full_data[full_data['order_dow'] != 6]

saturday_top = saturday_orders['product_name'].value_counts().head(10)
other_top = other_orders['product_name'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(10), saturday_top.values[::-1], color='coral')
plt.yticks(range(10), saturday_top.index[::-1])
plt.title('Top 10 Products Ordered on Saturdays', fontsize=14, fontweight='bold')
plt.xlabel('Number of Orders')
plt.tight_layout()
plt.show()

print("\nRecommendation: Emphasize these products on Saturdays:")
for i, (prod, count) in enumerate(saturday_top.items(), 1):
    print(f"{i}. {prod}: {count:,} orders")

---

## Question 2: What products are frequently ordered with chocolate?

In [ ]:
# Find orders containing chocolate
chocolate_orders = full_data[full_data['product_name'].str.contains('chocolate', case=False, na=False)]['order_id'].unique()
chocolate_order_products = full_data[full_data['order_id'].isin(chocolate_orders)]

# Products frequently ordered with chocolate (exclude chocolate itself)
co_purchased = chocolate_order_products[~chocolate_order_products['product_name'].str.contains('chocolate', case=False, na=False)]
co_purchased_top = co_purchased['product_name'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(10), co_purchased_top.values[::-1], color='chocolate')
plt.yticks(range(10), co_purchased_top.index[::-1])
plt.title('Top 10 Products Purchased with Chocolate', fontsize=14, fontweight='bold')
plt.xlabel('Co-purchase Count')
plt.tight_layout()
plt.show()

print(f"\nChocolate orders: {len(chocolate_orders):,}")
print("Recommendation: Bundle these products with chocolate:")
for i, (prod, count) in enumerate(co_purchased_top.items(), 1):
    print(f"{i}. {prod}: {count:,} times")

---

## Question 3: What are the main customer profiles?

In [ ]:
# Create user-level features
user_features = full_data.groupby('user_id').agg({
    'order_id': 'nunique',
    'product_id': 'count',
    'order_hour_of_day': 'mean',
    'reordered': 'mean'
}).reset_index()

user_features.columns = ['user_id', 'total_orders', 'total_products', 'avg_hour', 'reorder_rate']
user_features['avg_products_per_order'] = user_features['total_products'] / user_features['total_orders']

# Simple segmentation based on behavior
def segment_customer(row):
    if row['avg_hour'] < 6 or row['avg_hour'] > 22:
        return 'Midnight Shopper'
    elif row['total_orders'] <= 3:
        return 'Casual Buyer'
    elif row['avg_products_per_order'] > 15:
        return 'Cart Addict'
    elif row['reorder_rate'] > 0.8:
        return 'Loyal Customer'
    else:
        return 'Regular Shopper'

user_features['segment'] = user_features.apply(segment_customer, axis=1)

# Visualize segments
segment_counts = user_features['segment'].value_counts()

plt.figure(figsize=(10, 6))
colors = plt.cm.Set2(range(len(segment_counts)))
plt.bar(segment_counts.index, segment_counts.values, color=colors)
plt.title('Customer Segments Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Customer Segment')
plt.ylabel('Number of Users')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nCustomer Segments:")
for seg, count in segment_counts.items():
    print(f"- {seg}: {count:,} users ({count/len(user_features)*100:.1f}%)")

---

## Question 4: What is the organic food proportion for vegetables?

In [ ]:
# Filter vegetables
vegetables = full_data[full_data['department'] == 'produce']

organic_veg = vegetables[vegetables['product_name'].str.contains('organic', case=False, na=False)]
non_organic_veg = vegetables[~vegetables['product_name'].str.contains('organic', case=False, na=False)]

organic_count = len(organic_veg)
non_organic_count = len(non_organic_veg)
total_veg = organic_count + non_organic_count

plt.figure(figsize=(8, 8))
plt.pie([organic_count, non_organic_count], 
        labels=['Organic', 'Non-Organic'],
        autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c'],
        explode=(0.05, 0),
        startangle=90)
plt.title('Organic vs Non-Organic Produce Orders', fontsize=14, fontweight='bold')
plt.show()

print(f"\nOrganic produce orders: {organic_count:,} ({organic_count/total_veg*100:.1f}%)")
print(f"Non-organic produce orders: {non_organic_count:,} ({non_organic_count/total_veg*100:.1f}%)")

---

## Question 5: Which items do customers put in cart first?

In [ ]:
# Items added first to cart (add_to_cart_order == 1)
first_items = full_data[full_data['add_to_cart_order'] == 1]
first_items_top = first_items['product_name'].value_counts().head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(10), first_items_top.values[::-1], color='steelblue')
plt.yticks(range(10), first_items_top.index[::-1])
plt.title('Top 10 Products Added First to Cart', fontsize=14, fontweight='bold')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

print("\nProducts most often added first:")
for i, (prod, count) in enumerate(first_items_top.items(), 1):
    print(f"{i}. {prod}: {count:,} times")

---

## Question 6: Which aisles are highly correlated?

In [ ]:
# Create aisle-order matrix
aisle_orders = full_data.groupby(['order_id', 'aisle']).size().unstack(fill_value=0)

# Calculate correlation between aisles
aisle_corr = aisle_orders.corr()

# Get top correlated pairs (excluding self-correlation)
corr_pairs = []
for i in range(len(aisle_corr.columns)):
    for j in range(i+1, len(aisle_corr.columns)):
        corr_pairs.append({
            'aisle1': aisle_corr.columns[i],
            'aisle2': aisle_corr.columns[j],
            'correlation': aisle_corr.iloc[i, j]
        })

corr_df = pd.DataFrame(corr_pairs).sort_values('correlation', ascending=False).head(10)

plt.figure(figsize=(12, 6))
labels = corr_df['aisle1'] + ' + ' + corr_df['aisle2']
plt.barh(range(10), corr_df['correlation'].values[::-1], color='purple')
plt.yticks(range(10), labels[::-1])
plt.title('Top 10 Most Correlated Aisle Pairs', fontsize=14, fontweight='bold')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print("\nHighly correlated aisle pairs (for product placement):")
for _, row in corr_df.iterrows():
    print(f"  {row['aisle1']} + {row['aisle2']}: {row['correlation']:.3f}")

---

## Question 7: Is there a relationship between time since last order and reorder probability?

In [ ]:
# Filter out first orders (days_since_prior_order = -1)
valid_orders = full_data[full_data['days_since_prior_order'] >= 0]

reorder_by_days = valid_orders.groupby('days_since_prior_order')['reordered'].mean() * 100

plt.figure(figsize=(12, 5))
plt.plot(reorder_by_days.index, reorder_by_days.values, marker='o', color='teal', linewidth=2)
plt.title('Reorder Rate vs Days Since Prior Order', fontsize=14, fontweight='bold')
plt.xlabel('Days Since Prior Order')
plt.ylabel('Reorder Rate (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInsight:")
print(f"- Reorder rate is highest around {reorder_by_days.idxmax()} days")
print(f"- Average reorder rate: {valid_orders['reordered'].mean()*100:.1f}%")

---

## Question 8: Which aisle and department have the most/least products?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Most ordered aisles
aisle_orders = full_data['aisle'].value_counts()
top_aisles = aisle_orders.head(10)
bottom_aisles = aisle_orders.tail(5)

axes[0].barh(range(10), top_aisles.values[::-1], color='green')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(top_aisles.index[::-1])
axes[0].set_title('Top 10 Most Ordered Aisles', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Orders')

# Department orders
dept_orders = full_data['department'].value_counts()
axes[1].barh(range(len(dept_orders)), dept_orders.values[::-1], color='orange')
axes[1].set_yticks(range(len(dept_orders)))
axes[1].set_yticklabels(dept_orders.index[::-1])
axes[1].set_title('Orders by Department', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Orders')

plt.tight_layout()
plt.show()

print("\nAisles:")
print(f"  Most ordered: {top_aisles.index[0]} ({top_aisles.values[0]:,} orders)")
print(f"  Least ordered: {bottom_aisles.index[-1]} ({bottom_aisles.values[-1]:,} orders)")

print("\nDepartments:")
print(f"  Top: {dept_orders.index[0]} ({dept_orders.values[0]:,} orders)")
print(f"  Bottom: {dept_orders.index[-1]} ({dept_orders.values[-1]:,} orders)")

---

## Summary

In [ ]:
print("="*60)
print("BUSINESS INSIGHTS SUMMARY")
print("="*60)
print("""
Key Recommendations:

1. SATURDAY: Focus on top sellers (bananas, organic products)
2. CHOCOLATE: Bundle with milk, eggs, and bread products
3. CUSTOMERS: Target segments differently (Midnight, Cart Addicts)
4. ORGANIC: ~X% of produce is organic - growing trend
5. FIRST CART: Place top first-items near entrance
6. AISLE LAYOUT: Place correlated aisles nearby
7. REORDER: Optimize for 7-14 day reorder cycles
8. INVENTORY: Prioritize produce, dairy, and beverages
""")